#  Inference and Generation Demo

This notebook demonstrates text generation with the trained SLM model.

In [ ]:
import sys
sys.path.append('..')

import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, HTML
import ipywidgets as widgets
from pathlib import Path

from src.models import Gemma3Model, GEMMA3_CONFIG_270M
from src.data import Tokenizer
from src.inference import TextGenerator

%matplotlib inline
sns.set_style("whitegrid")

## 1. Load Model

In [ ]:
# Setup device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Load model
model_path = "../best_model_params.pt"
if Path(model_path).exists():
    print("Loading model...")
    model = Gemma3Model(GEMMA3_CONFIG_270M)
    model.load_state_dict(torch.load(model_path, map_location=device))
    model = model.to(device)
    model.eval()
    print("Model loaded successfully!")
else:
    print("Model not found. Please train the model first.")
    model = None

In [ ]:
# Initialize tokenizer and generator
if model is not None:
    tokenizer = Tokenizer()
    generator = TextGenerator(model, tokenizer=tokenizer, device=device)
    print("Generator initialized!")

## 2. Interactive Generation

In [ ]:
def generate_text(prompt, max_tokens=100, temperature=0.8, top_k=50, top_p=0.95):
    """Generate text with custom parameters."""
    if model is None:
        return "Model not loaded!"
    
    try:
        generated = generator.generate(
            prompt=prompt,
            max_new_tokens=max_tokens,
            temperature=temperature,
            top_k=top_k,
            top_p=top_p,
            do_sample=True
        )
        return generated
    except Exception as e:
        return f"Error: {str(e)}"

# Create interactive widget
style = {'description_width': 'initial'}

prompt_input = widgets.Textarea(
    value='Once upon a time',
    placeholder='Enter your prompt here...',
    description='Prompt:',
    style=style,
    layout=widgets.Layout(width='100%', height='100px')
)

max_tokens_slider = widgets.IntSlider(
    value=100,
    min=10,
    max=500,
    step=10,
    description='Max Tokens:',
    style=style,
    layout=widgets.Layout(width='50%')
)

temperature_slider = widgets.FloatSlider(
    value=0.8,
    min=0.1,
    max=2.0,
    step=0.1,
    description='Temperature:',
    style=style,
    layout=widgets.Layout(width='50%')
)

top_k_slider = widgets.IntSlider(
    value=50,
    min=0,
    max=100,
    step=5,
    description='Top-k:',
    style=style,
    layout=widgets.Layout(width='50%')
)

generate_button = widgets.Button(
    description='Generate Text',
    button_style='primary',
    layout=widgets.Layout(width='200px')
)

output_text = widgets.Output()

def on_generate_clicked(b):
    with output_text:
        output_text.clear_output()
        prompt = prompt_input.value
        max_tokens = max_tokens_slider.value
        temperature = temperature_slider.value
        top_k = top_k_slider.value
        
        print(f"Prompt: {prompt}")
        print("-"*60)
        
        generated = generate_text(prompt, max_tokens, temperature, top_k)
        print(f"Generated:\n{generated}")

generate_button.on_click(on_generate_clicked)

# Display widgets
display(prompt_input)
display(max_tokens_slider, temperature_slider, top_k_slider)
display(generate_button)
display(output_text)

## 3. Batch Generation Examples

In [ ]:
# Predefined prompts for batch generation
prompts = [
    "The future of artificial intelligence is",
    "In the year 2050, humanity will",
    "The secret to happiness is",
    "Once upon a time in a distant galaxy",
    "The greatest invention of all time is"
]

def batch_generate(prompts, **kwargs):
    results = []
    for i, prompt in enumerate(prompts):
        generated = generate_text(prompt, **kwargs)
        results.append((prompt, generated))
    return results

# Generate
if model is not None:
    results = batch_generate(prompts, max_tokens=50, temperature=0.8)
    
    # Display results
    for i, (prompt, generated) in enumerate(results):
        print(f"\n{'='*60}")
        print(f"Prompt {i+1}: {prompt}")
        print("-"*60)
        print(f"Generated: {generated}")
        print("="*60)

## 4. Temperature Comparison

In [ ]:
def compare_temperatures(prompt, temperatures=[0.2, 0.5, 0.8, 1.2]):
    results = {}
    for temp in temperatures:
        generated = generate_text(prompt, temperature=temp, max_tokens=100)
        results[temp] = generated
    return results

if model is not None:
    test_prompt = "The meaning of life is"
    temp_results = compare_temperatures(test_prompt)
    
    print(f"Prompt: {test_prompt}")
    print("="*60)
    for temp, text in temp_results.items():
        print(f"\nTemperature {temp}:")
        print("-"*50)
        print(text)
        print("-"*50)

## 5. Quality Analysis

Analyze the quality of generated text.

In [ ]:
def analyze_generation(text):
    """Analyze generated text quality."""
    stats = {}
    
    # Token count
    tokens = tokenizer.encode(text)
    stats['tokens'] = len(tokens)
    
    # Word count
    words = text.split()
    stats['words'] = len(words)
    
    # Unique words
    stats['unique_words'] = len(set(words))
    
    # Average word length
    stats['avg_word_length'] = np.mean([len(w) for w in words])
    
    # Repetition ratio
    stats['repetition_ratio'] = 1 - (stats['unique_words'] / stats['words']) if stats['words'] > 0 else 0
    
    return stats

if model is not None:
    sample_text = generate_text("The world of tomorrow", max_tokens=200, temperature=0.8)
    stats = analyze_generation(sample_text)
    
    print("Generated Text:")
    print("-"*60)
    print(sample_text)
    print("-"*60)
    print("\nStatistics:")
    for key, value in stats.items():
        print(f"  {key.replace('_', ' ').title()}: {value:.2f}" if isinstance(value, float) else f"  {key.replace('_', ' ').title()}: {value}")

## 6. Save Generated Examples

In [ ]:
# Save examples to file
def save_examples(prompts, filename='../logs/generated_examples.txt'):
    with open(filename, 'w') as f:
        f.write("="*60 + "\n")
        f.write("GENERATED TEXT EXAMPLES\n")
        f.write("="*60 + "\n\n")
        
        for i, prompt in enumerate(prompts):
            generated = generate_text(prompt, max_tokens=100, temperature=0.8)
            f.write(f"Example {i+1}:\n")
            f.write(f"Prompt: {prompt}\n")
            f.write("-"*60 + "\n")
            f.write(f"Generated: {generated}\n")
            f.write("\n" + "="*60 + "\n\n")
    
    print(f"Examples saved to {filename}")

if model is not None:
    save_prompts = [
        "A journey through time",
        "The mystery of consciousness",
        "A recipe for success",
        "The beauty of nature",
        "Lessons from history"
    ]
    save_examples(save_prompts)